# CSW Metadata Crawler

This notebook crawls a CSW (OGC Catalog Service for the Web) endpoint and downloads
all metadata records as XML files into a configurable output directory.

## Prerequisites

Install dependencies (if not already done):

```bash
pip install requests lxml
```

## Shared output directory (Docker / local)

The `OUTPUT_DIR` path below is configurable. By default it points to
`../data/csw_metadata` (at the repository root); this directory is excluded from git.

To mount it inside a Docker service, add the following to `docker-compose.yml`:
```yaml
volumes:
  - ./data/csw_metadata:/data/csw_metadata
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — edit these values as needed
# ─────────────────────────────────────────────────────────────────────────────

# Base URL of the CSW service (without query parameters)
CSW_URL = "https://www.geocatalogue.fr/geonetwork/0e3541fb-db45-4166-982f-ee8363896023/fre/csw"

# Output directory for XML files.
# Accepts an absolute or relative path (relative to this notebook).
# This directory is excluded from git.
# Absolute example: "/home/user/data/csw_metadata"
OUTPUT_DIR = "../data/csw_metadata"

# Number of records per page (most CSW servers accept ≤ 500)
PAGE_SIZE = 100

# Delay between pages (seconds) — avoids overloading the server
REQUEST_DELAY = 0.5

In [ ]:
%pip install -r ../requirements-dev.txt

In [ ]:
import time
import requests
import xml.etree.ElementTree as ET
from pathlib import Path

# Register namespaces for clean serialisation
NAMESPACES = {
    "csw":   "http://www.opengis.net/cat/csw/2.0.2",
    "gmd":   "http://www.isotc211.org/2005/gmd",
    "gco":   "http://www.isotc211.org/2005/gco",
    "gml":   "http://www.opengis.net/gml",
    "srv":   "http://www.isotc211.org/2005/srv",
    "xlink": "http://www.w3.org/1999/xlink",
    "xsi":   "http://www.w3.org/2001/XMLSchema-instance",
    "ows":   "http://www.opengis.net/ows",
}

for prefix, uri in NAMESPACES.items():
    ET.register_namespace(prefix, uri)

output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_path.resolve()}")

In [ ]:
# ── 1. Service check (GetCapabilities) ───────────────────────────────────────

resp = requests.get(
    CSW_URL,
    params={"SERVICE": "CSW", "VERSION": "2.0.2", "REQUEST": "GetCapabilities"},
    timeout=30,
)
resp.raise_for_status()

root_caps = ET.fromstring(resp.content)
service_id = root_caps.find(
    ".//{http://www.opengis.net/ows}Title"
    or ".//{http://www.opengis.net/cat/csw/2.0.2}Title"
)
title_text = service_id.text if service_id is not None else "(not found)"
print(f"CSW service available: {title_text}")
print(f"HTTP status: {resp.status_code}")

In [ ]:
# ── 2. Count total records ────────────────────────────────────────────────────

resp_hits = requests.get(
    CSW_URL,
    params={
        "SERVICE":    "CSW",
        "VERSION":    "2.0.2",
        "REQUEST":    "GetRecords",
        "RESULTTYPE": "hits",
        "TYPENAMES":  "gmd:MD_Metadata",
        "OUTPUTSCHEMA": "http://www.isotc211.org/2005/gmd",
        "NAMESPACE":  "xmlns(gmd=http://www.isotc211.org/2005/gmd)",
        "MAXRECORDS": 1,
    },
    timeout=30,
)
resp_hits.raise_for_status()

root_hits = ET.fromstring(resp_hits.content)
sr = root_hits.find(".//csw:SearchResults", NAMESPACES)
if sr is None:
    # fallback without namespace
    sr = root_hits.find(".//SearchResults")

total_records = int(sr.get("numberOfRecordsMatched", 0)) if sr is not None else 0
total_pages   = (total_records + PAGE_SIZE - 1) // PAGE_SIZE

print(f"Total records   : {total_records}")
print(f"Pages to fetch  : {total_pages} (page size: {PAGE_SIZE})")

In [ ]:
# ── 3. Paginated crawl and XML save ──────────────────────────────────────────
#
# NOTE: this GeoNetwork instance hard-caps responses at 800 records regardless
# of the total catalogue size. Date-range filtering is broken server-side
# (GeoNetwork bug: dates are backslash-escaped in the internal ES query).
# The crawl below fetches as many records as the server will return.

def fetch_page(csw_url: str, start: int, max_records: int) -> bytes:
    r = requests.get(
        csw_url,
        params={
            "SERVICE":        "CSW",
            "VERSION":        "2.0.2",
            "REQUEST":        "GetRecords",
            "RESULTTYPE":     "results",
            "TYPENAMES":      "gmd:MD_Metadata",
            "OUTPUTSCHEMA":   "http://www.isotc211.org/2005/gmd",
            "NAMESPACE":      "xmlns(gmd=http://www.isotc211.org/2005/gmd)",
            "ELEMENTSETNAME": "full",
            "STARTPOSITION":  start,
            "MAXRECORDS":     max_records,
        },
        timeout=60,
    )
    r.raise_for_status()
    return r.content


def parse_next_record(xml_bytes: bytes) -> int:
    """Return nextRecord from SearchResults, or 0 when the server signals end."""
    root = ET.fromstring(xml_bytes)
    sr = root.find(".//csw:SearchResults", NAMESPACES) or root.find(".//SearchResults")
    if sr is None:
        return 0
    return int(sr.get("nextRecord", 0))


def extract_and_save(xml_bytes: bytes, output_dir: Path, seen: set) -> tuple[int, int]:
    """Write each new MD_Metadata record to disk; skip already-seen fileIdentifiers.
    Returns (saved_count, duplicate_count).
    """
    root = ET.fromstring(xml_bytes)
    ns_gmd = "http://www.isotc211.org/2005/gmd"
    ns_gco  = "http://www.isotc211.org/2005/gco"

    saved = dupes = 0
    for i, rec in enumerate(root.findall(f".//{{{ns_gmd}}}MD_Metadata")):
        file_id_el = rec.find(f"{{{ns_gmd}}}fileIdentifier/{{{ns_gco}}}CharacterString")
        file_id = (
            file_id_el.text.strip()
            if file_id_el is not None and file_id_el.text
            else None
        )

        if file_id:
            safe_name = (
                file_id.replace("/", "_").replace("\\", "_")
                        .replace(":", "_").replace(" ", "_")
            )
        else:
            safe_name = f"record_{len(seen) + i:05d}_unknown_id"

        if safe_name in seen:
            dupes += 1
            continue
        seen.add(safe_name)

        out_file = output_dir / f"{safe_name}.xml"
        out_file.write_text(
            '<?xml version="1.0" encoding="UTF-8"?>\n' + ET.tostring(rec, encoding="unicode"),
            encoding="utf-8",
        )
        saved += 1

    return saved, dupes


# ─── Main loop ────────────────────────────────────────────────────────────────
seen_ids     = {f.stem for f in output_path.glob("*.xml")}  # resume support
total_saved  = 0
total_errors = 0
start_pos    = 1

print(f"Already on disk: {len(seen_ids)} records.\n")

while start_pos > 0 and start_pos <= total_records:
    end_pos = min(start_pos + PAGE_SIZE - 1, total_records)
    print(f"\rFetching {start_pos} → {end_pos} / {total_records}...", end="", flush=True)
    try:
        xml_bytes = fetch_page(CSW_URL, start=start_pos, max_records=PAGE_SIZE)
        saved, _ = extract_and_save(xml_bytes, output_path, seen_ids)
        total_saved += saved

        next_rec = parse_next_record(xml_bytes)
        if next_rec == 0:
            print(f"\nServer returned nextRecord=0 at position {start_pos} (server cap reached).")
            break
        start_pos = next_rec

    except requests.HTTPError as exc:
        print(f"\nHTTP error at position {start_pos}: {exc}")
        total_errors += 1
        start_pos += PAGE_SIZE
    except ET.ParseError as exc:
        print(f"\nXML parse error at position {start_pos}: {exc}")
        total_errors += 1
        start_pos += PAGE_SIZE

    time.sleep(REQUEST_DELAY)

print(f"\nDone!")
print(f"  Records saved : {total_saved}")
print(f"  Errors        : {total_errors}")
print(f"  Total on disk : {len(list(output_path.glob('*.xml')))}")
print(f"  Output dir    : {output_path.resolve()}")

In [ ]:
# ── 4. Quick verification of downloaded content ───────────────────────────────

xml_files = sorted(output_path.glob("*.xml"))
print(f"XML files present: {len(xml_files)}")

# Preview of the first 5 files
for f in xml_files[:5]:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.1f} KB)")